In [1]:
from Util import math_functions
from Util.Problems import Problem, solution
from Util.math_functions import sieve_eratosthenes, sieve_eratosthenes_odd, sieve_sundaram, \
    sieve_eratosthenes_segmented, sieve_eratosthenes_two_segments, sieve_eratosthenes_wheel


class P010(Problem):
    number = 10
    title = "Summation of Primes"
    description = """<p>The sum of the primes below $10$ is $2 + 3 + 5 + 7 = 17$.</p><p>Find the sum of all the primes below two million.</p>"""
    n= 2000000

In [2]:
p = P010()
p.describe()

## Problem 10: Summation of Primes

<p>The sum of the primes below $10$ is $2 + 3 + 5 + 7 = 17$.</p><p>Find the sum of all the primes below two million.</p>

### Solution notes

_Prime sieve:_ A prime sieve is an algorithm that can be used to generate all primes up to a number $n$.

For this problem, a few different sieves will be implemented, to test which is fastest.

The sieve of Eratosthenes was implemented earlier for another problem. It is a simple ancient sieve (250 BCE), which works by looping through all numbers $2$ to $\sqrt{n}$. For each number, if it has not yet been marked as non-prime, it is prime, and we mark all of its multiples up to $n$ as non-prime.

In [3]:
@solution(P010, max_tests= 100, first=True, make_fast=True, warmup_args=(P010.n,))
def eratosthenes(n):
    return sum(sieve_eratosthenes(n))

In [4]:
p.test_once("eratosthenes")

142913828922 found after a separate test in 5.286200 ms by eratosthenes (first)


The odd only version of the Eratosthenes sieve starts with an array where every even number is already marked as non-prime. This means the loop can start at 3 and increment 2 each iteration, making this a bit faster (though not as much faster as you would expect)

In [5]:
@solution(P010, max_tests= 100, make_fast=True, warmup_args=(P010.n,))
def eratosthenes_odd(n):
    return sum(sieve_eratosthenes_odd(n))

In [6]:
p.test_once("eratosthenes_odd")

142913828922 found after a separate test in 4.920400 ms by eratosthenes_odd


This is a vague algorithm which is similar to the odds only eratosthenes algorithm but in a more complex way. I implemented it for comparison, but will not put in the time to understand it, as I am quite sure there are faster methods to come.

In [7]:
@solution(P010, max_tests= 100, make_fast=True, warmup_args=(P010.n,))
def sundaram(n):
    return sum(sieve_sundaram(n))

In [8]:
p.test_once("sundaram")

142913828922 found after a separate test in 4.986200 ms by sundaram


The sieve of Atkin might be the fastest, but as that is dark magic, I will not be touching it yet. A fully optimised sieve of Eratosthenes should perform very similar to Atkin. The first step in optimising Eratosthenes is segmentation. This means dividing the all numbers $1 - n$ into segments of size $\sqrt{n}$. These segments can then be calculated one by one instead of all at once. This means only previously discovered primes will be multiplied and multiples sieved out. Though this will not make the sieve faster in general, it will decrease memory usage and therefore can become faster as $n$ grows larger. At this point, I also rewrote all sieves to use numpy arrays instead of boolean lists, which nearly doubled the speed of most sieves.

In [9]:
@solution(P010, max_tests= 100, make_fast=True, warmup_args=(P010.n,))
def eratosthenes_segmented(n):
    return sum(sieve_eratosthenes_segmented(n))

In [10]:
p.test_once("eratosthenes_segmented")

142913828922 found after a separate test in 6.107700 ms by eratosthenes_segmented


Out of interest, I wanted to see what happens when splitting the sieve into only 2 segments: the first segment to calculate the base primes, and the second segment which calculates the multiples. As the segmentation theoretically is no faster than no segmentation, it would stand to reason that this is faster than segmentation. As this is not the case, it seems $n$ is sufficiently large that the segmentation is making it faster, but it is slower than with no segmentation due to the overhead of creating and populating the segments. Once they have been created, regular segmentation is faster than two segments.

In [11]:
@solution(P010, max_tests= 100, make_fast=True, warmup_args=(P010.n,))
def eratosthenes_two_segmented(n):
    return sum(sieve_eratosthenes_two_segments(n))

In [12]:
p.test_once("eratosthenes_two_segmented")

142913828922 found after a separate test in 7.514100 ms by eratosthenes_two_segmented


There is an additional sieving method which builds a "wheel". This can be used to filter all of the numbers down to way fewer numbers to actually sieve. In this implementation, I build a wheel, and run a eratosthenes-like algorithm on the filtered result. I have found wheel_four to yield the best results. This means a wheel which is created based on the first 4 primes {2, 3, 5, 7} What follows is the more detailed explanation of wheels from my math_function file. For further documentation see that file, but this should already explain the concept

~~~
 The wheel will be a 2d array, with only the rows where primes can be remaining.
In the case of wheel one, the factors are: [2,3] so the wheel size is 2 * 3 = 6.
This means any number can be written in the form

 6k + 1, 6k +2, 6k + 3, 6k + 4, 6k + 5 or 6k

 Out of these options, all primes are in 6k + 1 and 6k + 5, as these are coprime
with our initial factors.

 In general, for a given set of prime factors to start with, the size of the wheel
is the product of those factors.

 The number of "blocks" of our number definitions we need to loop over are equal to
n // wheel_size

 The sections which contain the primes can be determined by finding the numbers
between 1 and wheel size which are not divisible by any of the original factors.
In our example, these are the remainders [1,5].

Out of our original way of writing all numbers, we now only need to check the
numbers in the form 6k + 1 and 6k + 5. Leaving us with a wheel of shape
(num_blocks, len(remainders)).

We then loop through the possible remainders. Within this loop, we loop through
k in the range 1 - num_blocks to check the possible prime numbers of the form
wheel_size * k + remainder
~~~

In [13]:
@solution(P010, max_tests= 100, make_fast=True, warmup_args=(P010.n,))
def eratosthenes_wheel_two(n):
    return sum(sieve_eratosthenes_wheel(n, 2))
@solution(P010, max_tests= 100, make_fast=True, warmup_args=(P010.n,))
def eratosthenes_wheel_three(n):
    return sum(sieve_eratosthenes_wheel(n, 3))
@solution(P010, max_tests= 100, best=True, make_fast=True, warmup_args=(P010.n,))
def eratosthenes_wheel_four(n):
    return sum(sieve_eratosthenes_wheel(n, 4))
@solution(P010, max_tests= 10, make_fast=True, warmup_args=(P010.n,))
def eratosthenes_wheel_five(n):
    return sum(sieve_eratosthenes_wheel(n, 5))

In [14]:
p.test_all()

142913828922 found after 100 tests in 5.129005 ms by eratosthenes (first)
142913828922 found after 100 tests in 5.014873 ms by eratosthenes_odd
142913828922 found after 100 tests in 6.170972 ms by eratosthenes_segmented
142913828922 found after 100 tests in 5.623010 ms by eratosthenes_two_segmented
142913828922 found after 10 tests in 11.353180 ms by eratosthenes_wheel_five
142913828922 found after 100 tests in 4.660222 ms by eratosthenes_wheel_four (best)
142913828922 found after 100 tests in 5.393406 ms by eratosthenes_wheel_three
142913828922 found after 100 tests in 7.255601 ms by eratosthenes_wheel_two
142913828922 found after 100 tests in 4.771475 ms by sundaram
